<p align="center"><a href="https://www.plus2net.com/python/pandas-dt-timestamp.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas Timestamp: Dates, Epochs and Timezones

Run cells in order. Requires Pandas only. All dates and tables are embedded synthetic practice data. Locale-specific names may differ. Try the exercise before the solution.

## Convert the original Unix epoch seconds

pd.Timestamp() constructs a scalar datetime. It is not DataFrame.timestamp(). For a numeric Unix epoch, unit is essential. Preserve the original epoch example and the <a href="../php_tutorial/php_time_stamp.php">PHP timestamp tool</a>.

In [ ]:
import pandas as pd
stamp = pd.Timestamp(1587099167, unit="s", tz="Asia/Kolkata")
print(stamp)
print(stamp.value)
assert stamp.strftime("%Y-%m-%d %H:%M:%S") == "2020-04-17 10:22:47"
assert stamp.value == 1587099167000000000

**Expected output**

```text
2020-04-17 10:22:47+05:30
1587099167000000000
```

## Construct from year, month and day

The current constructor supports calendar/time components, tz and unit. The old freq constructor argument is no longer supported. value gives epoch nanoseconds when representable; it can overflow outside that range.

In [ ]:
parts = pd.Timestamp(year=2020, month=12, day=23, hour=18, minute=20, second=5)
print(parts.day)
print(parts.asm8)
print(parts.dayofweek)
print(parts.daysinmonth)
assert parts.dayofweek == 2 and parts.daysinmonth == 31

**Expected output**

```text
23
2020-12-23T18:20:05.000000
2
31
```

## Construct from date text and inspect fields

Use an unambiguous ISO string. For parsing a full column with a format, use <a href="pandas-dt-to_datetime.php">to_datetime()</a>. asm8 resolution can differ between versions.

In [ ]:
from_text = pd.Timestamp("2020-12-23 18:20:05")
print(from_text.day)
print(from_text.asm8)
print(from_text.weekday())
print(from_text.dayofweek)
print(from_text.dayofyear)
assert from_text == parts and from_text.dayofyear == 358

**Expected output**

```text
23
2020-12-23T18:20:05.000000
2
2
358
```

## Get the current time

The original current-time concept is retained. Actual values change on each run; this example prints only stable properties so the notebook can be checked repeatedly.

In [ ]:
now = pd.Timestamp.now(tz="Asia/Kolkata")
print("Timezone:", str(now.tz))
print("Seconds valid:", 0 <= now.second <= 59)
print("Epoch positive:", now.timestamp() > 0)
assert str(now.tz) == "Asia/Kolkata"

**Expected output**

```text
Timezone: Asia/Kolkata
Seconds valid: True
Epoch positive: True
```

## Convert timezone and localise naive wall time

Localising assigns a timezone to a wall time; converting changes the display of the same instant. Explicit UTC removes ambiguity in epoch conversions.

In [ ]:
utc = stamp.tz_convert("UTC")
wall = pd.Timestamp("2020-04-17 10:22:47").tz_localize("Asia/Kolkata")
print(utc)
print("Epoch seconds:", utc.timestamp())
assert wall == stamp and utc.timestamp() == 1587099167.0

**Expected output**

```text
2020-04-17 04:52:47+00:00
Epoch seconds: 1587099167.0
```

## Convert the original daily date range to epoch seconds

The original integer division by one billion assumed nanosecond storage. Divide elapsed timedeltas by a one-second Timedelta instead. The source wall times are explicitly interpreted as UTC here.

In [ ]:
dates = pd.date_range("2020-12-18 18:15:05", periods=4, freq="D", tz="UTC")
seconds = (dates - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta(seconds=1)
print(seconds.tolist())
assert seconds.tolist() == [1608315305.0, 1608401705.0, 1608488105.0, 1608574505.0]

**Expected output**

```text
[1608315305.0, 1608401705.0, 1608488105.0, 1608574505.0]
```

## Calculate elapsed time for a synthetic event

Keep instants timezone aware and convert to text only for display. Calendar dates and elapsed durations answer different questions.

In [ ]:
start = pd.Timestamp("2026-09-05 09:00", tz="UTC")
finish = pd.Timestamp("2026-09-05 10:30", tz="UTC")
minutes = (finish - start) / pd.Timedelta(minutes=1)
print("Elapsed minutes:", minutes)
print("Local finish:", finish.tz_convert("Asia/Kolkata").strftime("%Y-%m-%d %H:%M %z"))
assert minutes == 90

**Expected output**

```text
Elapsed minutes: 90.0
Local finish: 2026-09-05 16:00 +0530
```

## Common mistakes

Use Timestamp with capital T. A Timestamp is a datetime object, not simply an epoch number. Use unit="s" for epoch seconds; do not confuse unit with units. Do not mix naive and aware timestamps in arithmetic. value and asm8 have different representation purposes. Use <a href="pandas-dt-strftime.php">strftime()</a> for display. For regional daylight-saving transitions, resolve ambiguous/nonexistent wall times explicitly.

## Exercise: epoch round trip

Convert epoch 0 to UTC, add one hour and verify its epoch seconds. Continue with <a href="pandas-dataframe-exercise3-2.php">Date and Time basics exercises</a>.

## Exercise solution

A one-hour duration is 3600 seconds.

In [ ]:
answer = pd.Timestamp(0, unit="s", tz="UTC") + pd.Timedelta(hours=1)
print(answer)
print(answer.timestamp())
assert answer.timestamp() == 3600

**Expected output**

```text
1970-01-01 01:00:00+00:00
3600.0
```